# Part I: Initial Benchmark
We are going to benchmark our current implementation, first we want to set $n$ and $m$ so that they can be on a reasonably sized instance. 

We use the ks_func function we implemented in HW2, and we set $n$  and $m$  to be 1,000,000 instead of 1000 to make the memory bottleneck and performance issues more observable.  And we use the benchmark tool to see the execute time and see what'll happen.



In [5]:
using Plots
using Revise
includet("C:/Users/Odysseus/GR6104_Homework/HW3/src/ks-stat.jl")

In [6]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000)
m = randn(1000)
@btime ks_func(n,m)
@benchmark ks_func(n,m)


  105.600 μs (13 allocations: 94.04 KiB)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  110.200 μs …  50.802 ms  ┊ GC (min … max): 0.00% … 99.10%
 Time  (median):     222.400 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   264.040 μs ± 934.982 μs  ┊ GC (mean ± σ):  9.70% ±  2.93%

    ▃█▁    ▂▁▁   ▅▂                                              
  ▁▁████▆▆▇████▆▇███▇▆▅▄▄▄▅▅▅▅▅▄▄▃▃▃▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  110 μs           Histogram: frequency by time          560 μs <

 Memory estimate: 94.04 KiB, allocs estimate: 13.

The current implementation takes approximately 170 ms to run. More importantly, it allocates 91.55 MiB of memory. This indicates a significant inefficiency, likely due to the creation of intermediate data structures during the labeling and sorting process.

# Part II: Iterative Code Optimization
## 1. Identify Bottlenecks

In [ ]:
ks_func(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n, m)

## Function and Line Number:
Based on the result from the Flame Graph, the primary bottleneck is located in the ks_func function, specifically at line 10 in ks-stat.jl. This line corresponds to the sort! operation on the combined array.

## Nature of the Bottleneck:
The bottleneck is primarily caused by unnecessary memory allocation, which leads to excessive computation:
    1. Memory Allocation caused by string, since the code creates a new tuple containing string (value,"label") for every single data point for a single run, it causes a huge volume of pressure on system's memory and the Collector.
    2. The Flame Graph shows that 74% of the execution time is spent in sort!. We might need to change the sort! a bit.

# 2. Alternative Implementations:

### Rationale for Alternatives
The bottleneck in the original code was the creation of intermediate `(value, label)` tuples and sorting a mixed array.
* **Alternative 1 (Separate Sorting + Two-Pointer):** Instead of mixing the arrays, we sort `X` and `Y` separately. This allows us to use Julia's highly optimized native float sorting. We then use a "Two-Pointer" algorithm to calculate the KS statistic in faster way which will avoid more allocations.

In [3]:
# Implementation for the alternatives two-pointer function 


function ks_func_2pt(X::AbstractVector,Y::AbstractVector)

    max_diff= 0
    cdf_x = 0
    cdf_y = 0
    n = length(X)
    m = length(Y)
    sort_x = sort(X)
    sort_y =sort(Y)
    i=1
    j=1

    while i<=n || j<=m 
        if i>n
            current_val = sort_y[j]
        elseif j>m
            current_val = sort_x[i]
        else 
            current_val = min(sort_x[i],sort_y[j])
        end
    

        while i <=n && sort_x[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sort_y[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff

end        

ks_func_2pt (generic function with 1 method)

# 3. Benchmark Alternatives:

In [2]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000)
m = randn(1000)
@btime ks_func_2pt(n,m)
@benchmark ks_func_2pt(n,m)

  20.300 μs (1 allocation: 16 bytes)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  22.000 μs …  1.040 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     44.600 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   45.521 μs ± 24.253 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃▂▁▂▁▄▅▄▃▃▅█▄▁▁▁                                            ▁
  ████████████████████▇▆▆▇▇▇▇▆▆▅▅▆▅▄▅▄▅▅▅▄▄▅▅▄▄▄▃▄▅▄▃▄▃▄▃▄▃▄▃ █
  22 μs        Histogram: log(frequency) by time       141 μs <

 Memory estimate: 16 bytes, allocs estimate: 1.

In [8]:
ks_func(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n, m)

┌ Warning: There were no samples collected.
│ Run your program longer (perhaps by running it multiple times),
│ or adjust the delay between samples with `Profile.init()`.
└ @ Profile C:\Users\Odysseus\.julia\juliaup\julia-1.12.4+0.x64.w64.mingw32\share\julia\stdlib\v1.12\Profile\src\Profile.jl:1367
